# Real Data Performance

This notebook compares the performance of the EM approach to LOOCV approach using real-world datasets.

## Preview Experiment

In [1]:
import numpy as np
import pandas as pd

from fastridge import RidgeEM, RidgeLOOCV
from experiments import run_real_data_experiments
from problems import EmpiricalDataProblem

problems = [
    EmpiricalDataProblem('abalone',  'Rings'),
    EmpiricalDataProblem('airfoil',  'scaled-sound-pressure'),
    # EmpiricalDataProblem('automobile', 'price'),              # ValueError
    EmpiricalDataProblem('concrete', 'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes', 'target'),
    # EmpiricalDataProblem('facebook', 'Total Interactions'),   # ValueError
    EmpiricalDataProblem('forest',   'area'),
    # EmpiricalDataProblem('parkinsons', 'motor_UPDRS', drop=['total_UPDRS']),  # expensive
    # EmpiricalDataProblem('real_estate', 'Y house price of unit area'),        # expensive
    EmpiricalDataProblem('student',  'G3'),
    EmpiricalDataProblem('yacht',    'Residuary_resistance'),   # much lower r2 (0.65 vs. 0.97)
    # EmpiricalDataProblem('crime', 'ViolentCrimesPerPop'),     # expensive
]

estimators = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results = run_real_data_experiments(problems, estimators, n_iterations=10, seed=1, verbose=True)
print()

rows = []
for problem, data_result in zip(problems, results):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = data_result['EM']['p']
    row['n_train']  = data_result['EM']['n_train']
    row['n:p']      = data_result['EM']['n_train'] / data_result['EM']['p']
    rows.append(row)
pd.DataFrame(rows).sort_values('n_train', ascending=False)

abalone (n=9, p=9)..........
airfoil (n=5, p=5)..........
concrete (n=8, p=8)..........
diabetes (n=10, p=10)..........
forest (n=27, p=27)..........
student (n=41, p=41)..........
yacht (n=6, p=6)..........



,dataset,target,EM,CV_glm,CV_fix,Speed-Up,p,n_train,n:p
0,abalone,Rings,0.535489,0.535628,0.535624,8.678167,9.0,2923,324.777778
1,airfoil,scaled-sound-pressure,0.516087,0.516071,0.516047,10.204450,5.0,1052,210.400000
2,concrete,Concrete compressive strength,0.588105,0.588173,0.588153,9.319847,8.0,721,90.125000
5,student,G3,0.831943,0.832265,0.832319,5.979444,41.0,454,11.073171
4,forest,area,-0.069535,-0.268804,-0.089488,2.640756,26.3,361,13.726236
3,diabetes,target,0.469150,0.468564,0.468363,8.139754,10.0,309,30.900000
6,yacht,Residuary_resistance,0.648437,0.649024,0.648801,7.944673,6.0,215,35.833333


## Full Experiment

In [2]:
problems_full = [
    EmpiricalDataProblem('abalone',          'Rings'),
    # EmpiricalDataProblem('automobile',     'price'),                           # ValueError because of N/A treatment
    # EmpiricalDataProblem('autompg',        'mpg'),                             # Source does not seem to work
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    # EmpiricalDataProblem('crime',          'ViolentCrimesPerPop'),             # SVM non-converging for some run
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    # EmpiricalDataProblem('facebook',       'Total Interactions'),              # ValueError because of N/A treatment
    EmpiricalDataProblem('forest',           'area'),
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']),
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3'),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results_full = run_real_data_experiments(problems_full, estimators_full,
                                         n_iterations=100, seed=1, verbose=True)
print()

rows_full = []
for problem, data_result in zip(problems_full, results_full):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_full.append(row)
pd.DataFrame(rows_full).sort_values('n_train', ascending=False)

abalone (n=9, p=9)....................................................................................................
airfoil (n=5, p=5)....................................................................................................
boston (n=13, p=13)....................................................................................................
concrete (n=8, p=8)....................................................................................................
diabetes (n=10, p=10)....................................................................................................
forest (n=27, p=27)....................................................................................................
naval_propulsion (n=16, p=16)....................................................................................................
naval_propulsion (n=16, p=16)....................................................................................................
parkinsons (n=19, p=

,dataset,target,EM,CV_glm,CV_fix,Speed Up Ratio,p,n_train,n:p
6,naval_propulsion,GT_compressor_decay,0.842063,0.841976,0.842064,7.010140,15.00,8353,556.866667
7,naval_propulsion,GT_turbine_decay,0.910962,0.910947,0.910962,6.935101,15.00,8353,556.866667
8,parkinsons,motor_UPDRS,0.144759,0.144714,0.144780,5.781275,19.00,4112,216.421053
9,parkinsons,total_UPDRS,0.168777,0.168661,0.168585,5.850250,19.00,4112,216.421053
0,abalone,Rings,0.525323,0.525345,0.525338,8.815485,9.00,2923,324.777778
1,airfoil,scaled-sound-pressure,0.508602,0.508597,0.508573,10.872848,5.00,1052,210.400000
3,concrete,Concrete compressive strength,0.597844,0.597858,0.597864,9.140892,8.00,721,90.125000
11,student,G3,0.834272,0.834545,0.834528,6.165427,41.00,454,11.073171
5,forest,area,-0.062845,-0.194991,-0.071467,2.630493,26.46,361,13.643235
2,boston,medv,0.707480,0.706891,0.706966,7.743802,13.00,354,27.230769
